<a href="https://colab.research.google.com/github/Neuro-Algorithm/multi-mouse-tracker/blob/main/Multi_Animal_Pose_Estimation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install --pre deeplabcut

In [ ]:
from pathlib import Path
import deeplabcut

In [ ]:
from google.colab import files

print("Upload your mouse video file:")
uploaded = files.upload()

for filepath, content in uploaded.items():
    print(f' Uploaded: "{filepath}" ({len(content):,} bytes)')
    video_path = Path(filepath).resolve()

In [ ]:
superanimal_name = "superanimal_quadruped"
model_name = "hrnet_w32"
detector_name = "fasterrcnn_resnet50_fpn_v2"
pcutoff = 0.15

print(f"\n Using model: {superanimal_name}")
print(f" Network: {model_name}")
print(f" Detector: {detector_name}")
print(f" Confidence cutoff: {pcutoff}")

In [ ]:
print("\n Analyzing video with SuperAnimal...")
print("This may take a few minutes...")

deeplabcut.video_inference_superanimal(
    [video_path],
    superanimal_name,
    model_name=model_name,
    detector_name=detector_name,
    videotype="mp4",
    video_adapt=False,
    scale_list=[250],
    pcutoff=pcutoff,
)

print("Analysis complete!")

In [ ]:
videotype = video_path.suffix
scale_list = []

deeplabcut.video_inference_superanimal(
    [video_path],
    superanimal_name,
    model_name=model_name,
    detector_name=detector_name,
    videotype=videotype,
    video_adapt=False,
    scale_list=scale_list,
    pcutoff=pcutoff,
)

In [ ]:
from base64 import b64encode
from IPython.display import HTML
import glob


In [ ]:
import pandas as pd

# Define the h5 file path
h5_file = '/content/Laboratory Rat Model (Wistar rats)_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_.h5'

# Load the data
df = pd.read_hdf(h5_file)

# Get frame data
frame_data = df.iloc[0]

# Extract animal0 data
animal_data = frame_data['superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_']['animal0']

# See what bodyparts are available
print("Available bodyparts:")
for bp in animal_data.index.get_level_values(0).unique():
    print(f"  - {bp}")

In [ ]:
import pandas as pd
import cv2
import matplotlib.pyplot as plt

# Define paths
h5_file = '/content/Laboratory Rat Model (Wistar rats)_superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_.h5'
video_path = '/content/Laboratory Rat Model (Wistar rats).mp4'

# Load predictions
df = pd.read_hdf(h5_file)

# Load a frame from the video
cap = cv2.VideoCapture(video_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 100)  # Frame 100
ret, frame = cap.read()
cap.release()

# Get predictions for frame 100
frame_data = df.iloc[100]

# Extract animal0 data
animal_data = frame_data['superanimal_quadruped_hrnet_w32_fasterrcnn_resnet50_fpn_v2_']['animal0']

# Plot keypoints with high confidence
plt.figure(figsize=(16, 10))
plt.imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

# Define bodyparts to plot (head, ears, body, paws, knees)
bodyparts_to_plot = [
    # Head
    'nose', 'upper_jaw', 'lower_jaw',
    'left_eye', 'right_eye',

    # Ears
    'left_earbase', 'left_earend',
    'right_earbase', 'right_earend',

    # Body middle
    'body_middle_left', 'body_middle_right',

    # Front legs
    'front_left_knee', 'front_left_paw',
    'front_right_knee', 'front_right_paw',

    # Back legs
    'back_left_knee', 'back_left_paw',
    'back_right_knee', 'back_right_paw'
]

# Define colors for different body parts
colors = {
    # Head
    'nose': 'red',
    'upper_jaw': 'yellow',
    'lower_jaw': 'orange',
    'left_eye': 'lime',
    'right_eye': 'lime',

    # Ears
    'left_earbase': 'magenta',
    'left_earend': 'magenta',
    'right_earbase': 'cyan',
    'right_earend': 'cyan',


    # Front legs (green shades)
    'front_left_knee': 'springgreen',
    'front_left_paw': 'lime',
    'front_right_knee': 'springgreen',
    'front_right_paw': 'lime',

    # Back legs (purple shades)
    'back_left_knee': 'violet',
    'back_left_paw': 'green',
    'back_right_knee': 'violet',
    'back_right_paw': 'green'
}

for bodypart in bodyparts_to_plot:
    x = animal_data[bodypart]['x']
    y = animal_data[bodypart]['y']
    likelihood = animal_data[bodypart]['likelihood']

    if likelihood > 0.5:  # Only show confident predictions
        color = colors.get(bodypart, 'white')

        # Bigger markers for ears and paws
        if 'ear' in bodypart or 'paw' in bodypart:
            markersize = 9
        else:
            markersize = 11

        plt.plot(x, y, 'o', color=color, markersize=markersize,
                markeredgecolor='white', markeredgewidth=2)
        plt.text(x+8, y-8, bodypart, fontsize=8, color=color,
                weight='bold', bbox=dict(boxstyle='round', facecolor='black', alpha=0.7))

plt.title('Full Mouse Body Keypoints', fontsize=16, weight='bold')
plt.axis('off')
plt.tight_layout()
plt.show()

# Print confidences organized by body region
print("KEYPOINT CONFIDENCES")

print("\nEARS:")
print(f"  Left ear base:  {animal_data['left_earbase']['likelihood']:.3f}")
print(f"  Left ear end:   {animal_data['left_earend']['likelihood']:.3f}")
print(f"  Right ear base: {animal_data['right_earbase']['likelihood']:.3f}")
print(f"  Right ear end:  {animal_data['right_earend']['likelihood']:.3f}")

print("\nBODY MIDDLE:")
print(f"  Left:  {animal_data['body_middle_left']['likelihood']:.3f}")
print(f"  Right: {animal_data['body_middle_right']['likelihood']:.3f}")

print("\nFRONT LEGS:")
print(f"  Left knee:  {animal_data['front_left_knee']['likelihood']:.3f}")
print(f"  Left paw:   {animal_data['front_left_paw']['likelihood']:.3f}")
print(f"  Right knee: {animal_data['front_right_knee']['likelihood']:.3f}")
print(f"  Right paw:  {animal_data['front_right_paw']['likelihood']:.3f}")

print("\nBACK LEGS:")
print(f"  Left knee:  {animal_data['back_left_knee']['likelihood']:.3f}")
print(f"  Left paw:   {animal_data['back_left_paw']['likelihood']:.3f}")
print(f"  Right knee: {animal_data['back_right_knee']['likelihood']:.3f}")
print(f"  Right paw:  {animal_data['back_right_paw']['likelihood']:.3f}")
